# 01. Extracellular stimulation and recording

In this tutorial, we will explore how to use AxonML to simulate extracellular stimulation and recording in neuronal models. We will cover the following topics:

- Computing extracellular stimulation potentials
- Applying extracellular stimulation to neuronal populations
- Recording extracellular potentials from neuronal populations


## General principles

AxonML treats extracellular stimulation and recording in terms of the **extracellular potential** experienced by each model compartment. Conceptually, you should think of there being a value
$\phi_e(x_i, t_n)$
for **every compartment** $x_i$ and **every timestep** $t_n$. 

The stimulation machinery in AxonML is essentially about specifying and evaluating this field $\phi_e(x, t)$ over space and time, then feeding those values into the cable equations (which may themselves include multiple extracellular layers, e.g. for myelinated fibers).

### Quasistatic fields and separation of space and time

In typical neural-interface settings, electrode currents are relatively low frequency (up to a few kHz) and the spatial scale of interest is small compared to the speed of electromagnetic propagation. Under these conditions, the **quasistatic approximation** is appropriate: the spatial pattern of the extracellular potential adjusts essentially instantaneously to changes in electrode current, and displacement currents can be neglected.

Practically, this means we can separate the field into:

- a **spatial component** (a “lead field”) that depends only on position, and  
- a **temporal component** that depends only on the stimulation waveform.

For a single stimulating electrode, we can write
```{math}
\phi_e(x, t) \approx I(t)\,\Phi(x),
```
where:

- $\Phi(x)$ is the extracellular potential distribution produced by a **unit** current at that electrode (from an analytic solution or FEM), and  
- $I(t)$ is the time-varying current waveform delivered by the stimulator.

For multiple electrodes, this generalizes to
```{math}
\phi_e(x, t) = \sum_e I_e(t)\,\Phi_e(x),
```
where each $\Phi_e(x)$ is the unit-current field for electrode $e$.

In AxonML, this is exactly how **field objects** are used: you provide a spatial solution (or analytic expression) for unit input, and then drive it with one or more time-varying waveforms. Internally, AxonML evaluates $\phi_e(x_i, t_n)$ for every compartment and timestep without requiring you to recompute the field solution when you change the waveform.

### Reciprocity and extracellular recording

The same volume-conductor physics applies to extracellular recording. Under linear, quasistatic conditions, we can use **reciprocity**: the sensitivity of a recording electrode to a given membrane current source is the same as the extracellular potential that source would experience if we injected unit current at the recording site.

If $\Phi_{\text{rec}}(x)$ denotes the potential field generated by a **unit current at the recording electrode**, then the recorded signal can be written schematically as
```{math}
V_{\text{rec}}(t) = \sum_i \Phi_{\text{rec}}(x_i)\, I_m(x_i, t),
```
where $I_m(x_i, t)$ is the transmembrane current at compartment $x_i$.

In AxonML, this corresponds to using a “recording lead field” to weight the membrane currents from each compartment and sum them into a simulated local field potential or extracellular trace.

In summary:

- **Stimulation** is modeled by **imposing** $\phi_e(x_i, t_n)$ per compartment and timestep, using precomputed spatial fields scaled by time-varying waveforms.  
- **Recording** is modeled by **reading out** a weighted sum of membrane currents, using reciprocity-based lead fields for the recording electrode.

The rest of this tutorial will show how to construct these fields, attach them to neuronal populations, and use AxonML to simulate both extracellular stimulation and recording.


## Example: Extracellular kHz stimulation and recording

We will simulate a single myelinated axon model receiving extracellular kHz stimulation from a point source electrode, while recording the extracellular potential at another nearby point.